# 逐层点时开发账户现金与SOXL相对选择

只读1056日f696+b472原生Qlib调整价账户。按09:47/10:32/收盘/次开盘实际仓位与Tiered费用分段闭合；相对同价SOXX做描述性对数分量，10个恢复日列残差。不认证独立分红现金或合同v2账户。

In [1]:
import os,json,hashlib
from pathlib import Path
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"]="1"
root=Path.cwd();card=root/"research/stage1_v5_oof_cash_selection_components_20261030"
if (card/"summary.json").exists():
    import qlib
    from src.core.recorder_sources import load_verified_recorder_objects
    summary=json.loads((card/"summary.json").read_text())
    assert summary["plan_sha256"]==hashlib.sha256((card/"plan.json").read_bytes()).hexdigest()
    qlib.init(provider_uri=str(root/"data/qlib/full_path_tiered_allocation_20261013"),region="us",kernels=1)
    saved,receipt=load_verified_recorder_objects(card,expected_recorder_id=summary["recorder_id"],
        artifacts=["analysis/daily","analysis/quarter","analysis/rolling126","analysis/summary"],
        expected_manifest_sha256=hashlib.sha256((card/"verification_manifest.json").read_bytes()).hexdigest())
    assert len(saved["analysis/daily"])==1056 and len(saved["analysis/rolling126"])==931
    print("verified_existing_development_component_recorder",summary["recorder_id"])
else:
    import os
    os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
    os.environ["MLFLOW_DISABLE_AGENT_HINT"]="1"
    import json,hashlib,warnings
    from pathlib import Path
    import numpy as np
    import pandas as pd
    import qlib
    from qlib.workflow import R
    from src.core.recorder_sources import load_verified_recorder_objects
    warnings.filterwarnings("ignore",message="The 'generic' unit for NumPy timedelta")
    root=Path.cwd();card=root/"research/stage1_v5_oof_cash_selection_components_20261030";plan_bytes=(card/"plan.json").read_bytes();plan=json.loads(plan_bytes)
    sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
    for role in ("parent_source","frame_source"):
     spec=plan[role];source=root/spec["card"]
     assert sha(source/"summary.json")==spec["summary_sha256"] and sha(source/"verification_manifest.json")==spec["manifest_sha256"]
     assert json.loads((source/"nas_binding.json").read_text())["evidence"]["sha256"]==spec["nas_archive_sha256"]
    assert sha(root/plan["provider_uri"]/"qlib_store_manifest.json")==plan["provider_manifest_sha256"]
    qlib.init(provider_uri=str(root/plan["provider_uri"]),region="us",kernels=1)
    p=plan["parent_source"];f=plan["frame_source"]
    parent,parent_receipt=load_verified_recorder_objects(root/p["card"],expected_recorder_id=p["recorder_id"],artifacts=["analysis/native/dated_f696_0"],expected_manifest_sha256=p["manifest_sha256"])
    source,frame_receipt=load_verified_recorder_objects(root/f["card"],expected_recorder_id=f["recorder_id"],artifacts=["inputs/covered_frame"],expected_manifest_sha256=f["manifest_sha256"])
    book=parent["analysis/native/dated_f696_0"];frame=source["inputs/covered_frame"].reset_index(drop=True);path=book["independent_replay"].reset_index(drop=True)
    assert len(frame)==len(path)==plan["expected_sessions"] and frame.trade_date.equals(path.trade_date)
    assert book["audit"]["status"]=="passed" and book["audit"]["slippage_bps"]==0
    assert pd.to_datetime(frame.feature_available_at).lt(pd.to_datetime(frame.decision_at)).all()
    positions=book["native_positions"];orders=book["orders"]
    by_day={pd.Timestamp(d):part for d,part in orders.groupby("trade_date",sort=True)}
    
    def q_of(position):
     return np.array([position.get_stock_amount(s) if position.check_stock(s) else 0. for s in ("SOXL","SOXX")],float)
    
    rows=[];max_nav_error=0.
    for i,row in frame.iterrows():
     day=pd.Timestamp(row.trade_date);opening_stamp=day+pd.Timedelta(hours=9,minutes=47);recovery_stamp=day+pd.Timedelta(hours=10,minutes=32)
     before=positions.get(opening_stamp);after=positions.get(recovery_stamp)
     if before is None or after is None:raise ValueError("blocked: missing native intra-session position")
     q0,q1=q_of(before),q_of(after)
     prices=np.vstack([row[["soxl_open","soxx_open"]].to_numpy(float),row[["soxl_recovery","soxx_recovery"]].to_numpy(float),
                       row[["close_soxl","close_soxx"]].to_numpy(float),row[["soxl_exit_open","soxx_exit_open"]].to_numpy(float)])
     pnl_assets=q0*(prices[1]-prices[0])+q1*(prices[2]-prices[1])+q1*(prices[3]-prices[2])
     fee=float(by_day.get(day,pd.DataFrame()).total.sum()) if day in by_day else 0.
     nav_end=float(path.loc[i,"native_expected_nav"]);r_port=float(path.loc[i,"strategy_return"]);opening_nav=nav_end/(1+r_port)
     err=opening_nav+float(pnl_assets.sum())-fee-nav_end
     max_nav_error=max(max_nav_error,abs(err))
     if abs(err)>1e-7:raise ValueError("blocked: segment P&L/fees do not reproduce native NAV")
     r_soxx=float(prices[3,1]/prices[0,1]-1.)
     if r_soxx<=-1 or r_port<=-1:raise ValueError("blocked: invalid return")
     cash_weight=float(before.get_cash())/opening_nav
     soxl_weight=float(q0[0]*prices[0,0]/opening_nav)
     cash=-cash_weight*r_soxx
     selection=float(pnl_assets[0]/opening_nav)-soxl_weight*r_soxx
     direct_fee=-fee/opening_nav
     arithmetic=r_port-r_soxx
     remainder=arithmetic-cash-selection-direct_fee
     paired=float(np.log1p(r_port)-np.log1p(r_soxx))
     scale=paired/arithmetic if abs(arithmetic)>1e-12 else 1/(1+r_soxx)
     rows.append({"signal_date":row.signal_date,"trade_date":day,"exit_date":row.exit_date,"year":int(day.year),"quarter":str(day.to_period("Q")),
        "paired_log_excess":paired,"cash_log":cash*scale,"soxl_selection_log":selection*scale,"direct_fees_log":direct_fee*scale,"recovery_and_timing_residual_log":remainder*scale,
        "opening_soxl_weight":soxl_weight,"opening_cash_weight":cash_weight,"raw_soxl_weight":float(row.soxl_weight),"raw_soxx_weight":float(row.soxx_weight),"raw_cash_weight":float(row.cash_weight),
        "alpha_probability":float(row.alpha_probability),"tail_probability":float(row.tail_probability),"risk_state":int(path.loc[i,"risk_state"]),"recovery_action":bool(path.loc[i,"recovery_action"])})
    daily=pd.DataFrame(rows)
    cols=["cash_log","soxl_selection_log","direct_fees_log","recovery_and_timing_residual_log"]
    closure=float(np.max(np.abs(daily[cols].sum(axis=1)-daily.paired_log_excess)))
    assert closure<1e-12 and max_nav_error<1e-7
    
    def describe(g,kind,period):
     result={"kind":kind,"period":str(period),"sessions":len(g),"first_trade":str(pd.Timestamp(g.trade_date.iloc[0]).date()),"last_exit":str(pd.Timestamp(g.exit_date.iloc[-1]).date())}
     for name in ["paired_log_excess",*cols]:result[name]=float(g[name].sum())
     result["relative_wealth_vs_adjusted_soxx"]=float(np.exp(result["paired_log_excess"]))
     result["mean_raw_soxl_weight"]=float(g.raw_soxl_weight.mean());result["mean_opening_cash_weight"]=float(g.opening_cash_weight.mean())
     result["recovery_days"]=int(g.recovery_action.sum())
     return result
    annual=pd.DataFrame([describe(g,"year",y) for y,g in daily.groupby("year",sort=True)])
    quarter=pd.DataFrame([describe(g,"quarter",q) for q,g in daily.groupby("quarter",sort=True)])
    rolling=pd.DataFrame([describe(daily.iloc[i:i+126],"rolling126",i) for i in range(len(daily)-125)])
    assert len(rolling)==plan["expected_rolling126"]
    annual.to_csv(card/"annual_components.csv",index=False);quarter.to_csv(card/"quarter_components.csv",index=False)
    weak=rolling.loc[rolling.paired_log_excess.idxmin()].to_dict();good=rolling.loc[rolling.paired_log_excess.idxmax()].to_dict()
    counts={"both_negative":int((rolling.cash_log.lt(0)&rolling.soxl_selection_log.lt(0)).sum()),
            "cash_negative_only":int((rolling.cash_log.lt(0)&rolling.soxl_selection_log.ge(0)).sum()),
            "selection_negative_only":int((rolling.cash_log.ge(0)&rolling.soxl_selection_log.lt(0)).sum()),
            "both_nonnegative":int((rolling.cash_log.ge(0)&rolling.soxl_selection_log.ge(0)).sum())}
    assert sum(counts.values())==len(rolling)
    (card/"window_extremes.json").write_text(json.dumps({"worst":weak,"best":good,"joint_sign_window_counts":counts},ensure_ascii=False,indent=2)+"\n")
    tracking=card/"qlib_recorder"
    with R.start(experiment_name=card.name,uri=str(tracking)):
     rec=R.get_recorder();rec.save_objects(artifact_path="inputs",plan=plan,source_audits={"parent":parent_receipt,"frame":frame_receipt})
     rec.save_objects(artifact_path="analysis",daily=daily,annual=annual,quarter=quarter,rolling126=rolling,window_extremes={"worst":weak,"best":good,"counts":counts})
     rec.log_metrics(max_native_nav_error_usd=max_nav_error,max_log_component_closure=closure,adjusted_soxx_relative_wealth=float(np.exp(daily.paired_log_excess.sum())))
    summary={"experiment":plan["experiment"],"plan_sha256":sha(card/"plan.json"),"decision":"read_only_oof_cash_selection_components_no_policy",
     "sessions":len(daily),"quarters":len(quarter),"rolling126_windows":len(rolling),"full_relative_wealth_vs_adjusted_soxx":float(np.exp(daily.paired_log_excess.sum())),
     "full_components":{name:float(daily[name].sum()) for name in cols},"max_native_nav_error_usd":max_nav_error,"max_log_component_closure":closure,
     "worst_window":weak,"best_window":good,"joint_sign_window_counts":counts,"new_model_fits":0,"new_native_accounts":0,"historical_research_passed":False,
     "recorder_id":rec.id,"experiment_id":str(rec.experiment_id),"recorder_uri":str(tracking),"recorder_local_dir":str(tracking/str(rec.experiment_id)/rec.id),
     "artifact_policy":{"policy":"slim_card_recorder_first","workflow_config":"workflow.yaml","recorder_only_artifacts":["inputs/*","analysis/*"]},
     "limitations":["Development adjusted-price plane does not separately certify dividend receivable cash flows or exact contract-v2 SOXX total-return benchmark.","Cash/selection allocation is descriptive starting-holdings attribution; ten recovery days enter the residual.","Rolling windows overlap and future outcome groups are not decision-time switches."]}
    (card/"summary.json").write_text(json.dumps(summary,ensure_ascii=False,indent=2)+"\n")
    print(json.dumps({"decision":summary["decision"],"full_components":summary["full_components"],"annual":annual.to_dict("records"),"worst":weak,"best":good,"counts":counts,"max_nav_error":max_nav_error},ensure_ascii=False))

## 中文结论

逐层点时1056日Qlib调整价账户以09:47/10:32/收盘/次开盘真实分段持仓和Tiered费用闭合，最大NAV差1.16e-10美元；现金/SOXL选择/费用/恢复残差四项对数闭合1.39e-17。开发期相对同价SOXX财富1.2501，现金+0.00329、选择+0.19780、费用−0.00569、恢复/时点残差+0.02779；10个恢复日在2020，不能称常规alpha。最弱126日2020-03-19至09-17现金−0.24045主导，SOXL选择+0.01746；最强2022-03-31至09-30现金保护+0.28019而选择−0.02872。全部931窗口同负94、只现金负522、只选择负279、均非负36；高度重叠，事后分组不能做开关。此为调整价开发诊断，不认证独立分红现金或合同v2原价SOXX门；0新拟合/账户。NAS上传及选定对象远端读回通过。下一步找能在决策前同时判断跨日现金与SOXL/SOXX相对价值的不同信息，否则不拟合。